In [1]:
import pickle
import pandas as pd

In [2]:
path = f'prepared_data/dataset_for_feature_engineering.pickle'

with open(path, 'rb') as f:
    df = pickle.load(f)

In [3]:
df["date"] = pd.to_datetime(df["date"])

In [4]:
df = df.sort_values(["ticker","date"]).reset_index(drop=True)

In [5]:
# Доходность — это процентное изменение цены акции за выбранный период времени.
# Она показывает, насколько выросла или упала цена по сравнению с прошлым значением.

# Экономический смысл:
# Если доходность положительная — акция растёт.
# Если отрицательная — акция падает.
# Чем больше модуль доходности, тем сильнее движение цены.

df["ret_1"]  = df.groupby("ticker")["close"].pct_change(1)   # дневное изменение цены
df["ret_3"]  = df.groupby("ticker")["close"].pct_change(1)   # изменение цены за три дня
df["ret_5"]  = df.groupby("ticker")["close"].pct_change(5)   # изменение за торговую неделю
df["ret_20"] = df.groupby("ticker")["close"].pct_change(20)  # изменение за торговый месяц

In [6]:
# Волатильность показывает, насколько нестабильно ведёт себя цена акции.
# Чем выше волатильность — тем выше риск и тем более резкие движения цены происходят.
#
# Экономический смысл:
# Высокая волатильность означает нервный, неустойчивый рынок.
# Низкая — спокойное, стабильное движение.

df["vol3"]  = df.groupby("ticker")["ret_1"].rolling(3).std().reset_index(0, drop=True)   # волатильность за 3 дня
df["vol_5"]  = df.groupby("ticker")["ret_1"].rolling(5).std().reset_index(0, drop=True)   # волатильность за 5 дней
df["vol_15"] = df.groupby("ticker")["ret_1"].rolling(15).std().reset_index(0, drop=True) # волатильность за 15 дней
df["vol_30"] = df.groupby("ticker")["ret_1"].rolling(30).std().reset_index(0, drop=True) # волатильность за 30 дней
df["vol_45"] = df.groupby("ticker")["ret_1"].rolling(45).std().reset_index(0, drop=True) # волатильность за 45 дней
df["vol_90"] = df.groupby("ticker")["ret_1"].rolling(90).std().reset_index(0, drop=True) # волатильность за 90 дней

In [7]:
# Моментум показывает, насколько сильно и в каком направлении двигалась цена акции
# за выбранный период времени.
#
# Экономический смысл:
# Положительный моментум означает, что акция находится в фазе роста.
# Отрицательный — что акция находится в фазе падения.

df["mom_3"]  = df.groupby("ticker")["close"].pct_change(3)   # сила движения за 3 дня
df["mom_5"]  = df.groupby("ticker")["close"].pct_change(5)   # сила движения за 5 дней
df["mom_15"]  = df.groupby("ticker")["close"].pct_change(15)   # сила движения за 5 дней
df["mom_20"] = df.groupby("ticker")["close"].pct_change(20)  # сила движения за 20 дней

In [8]:
# Скользящие средние показывают сглаженное направление движения цены акции.
# Они позволяют отделить устойчивый тренд от случайных рыночных колебаний.
#
# Экономический смысл:
# Если короткая средняя выше длинной — тренд восходящий.
# Если ниже — нисходящий.

df["sma_3"]  = df.groupby("ticker")["close"].transform(lambda x: x.rolling(3).mean())    # средняя цена за 3 дня
df["sma_5"]  = df.groupby("ticker")["close"].transform(lambda x: x.rolling(5).mean())    # средняя цена за 5 дней
df["sma_10"]  = df.groupby("ticker")["close"].transform(lambda x: x.rolling(10).mean())    # средняя цена за 5 дней
df["sma_20"] = df.groupby("ticker")["close"].transform(lambda x: x.rolling(20).mean())   # средняя цена за 20 дней

df["sma_ratio"] = df["sma_5"] / df["sma_20"] - 1   # относительное положение краткой и длинной средних


In [9]:
# Объём торгов показывает активность участников рынка.
# Он отражает, насколько активно инвесторы покупают и продают акцию.
#
# Экономический смысл:
# Рост объёма подтверждает силу движения цены.
# Падение объёма — признак ослабления интереса к акции.

df["vol_mean_5"] = df.groupby("ticker")["volume"].transform(lambda x: x.rolling(5).mean())  # средний объём за 5 дней
df["vol_mean_15"] = df.groupby("ticker")["volume"].transform(lambda x: x.rolling(15).mean())  # средний объём за 15 дней
df["vol_mean_45"] = df.groupby("ticker")["volume"].transform(lambda x: x.rolling(45).mean())  # средний объём за 45 дней
df["vol_ratio"]  = df["volume"] / df["vol_mean_5"]                                          # относительный объём


In [10]:
# Свечной диапазон показывает внутридневную амплитуду колебаний цены.
# Он отражает уровень рыночной неопределённости и борьбу покупателей и продавцов.

df["hl_range"] = (df["high"] - df["low"]) / df["close"]   # относительный дневной диапазон


In [11]:
# Временные признаки отражают календарную структуру торговых сессий.
# Рынок ведёт себя по-разному в разные дни недели и месяцы года из-за поведения инвесторов,
# налоговых периодов, отчётностей и ребалансировок фондов.

df["dow"] = df["date"].dt.weekday       # день недели (0 = понедельник, 4 = пятница)
df["month"] = df["date"].dt.month       # номер месяца (1–12)
df["week"] = df["date"].dt.isocalendar().week.astype(int)   # номер недели года
df["is_month_end"] = df["date"].dt.is_month_end.astype(int) # конец месяца (1 = да)
df["is_month_start"] = df["date"].dt.is_month_start.astype(int) # начало месяца

In [12]:
# Целевая переменная показывает, вырастет ли цена акции на следующий торговый день.
# 1 — если завтрашняя цена закрытия выше сегодняшней.
# 0 — если цена не выросла.

df["target"] = (df.groupby("ticker")["close"].shift(-1) > df["close"]).astype(int)

In [13]:
info = ['date', 'ticker']

In [14]:
df.drop(columns=info).corr()['target'].sort_values()

vol_ratio        -0.016887
volume           -0.008870
month            -0.007964
vol_mean_5       -0.006716
week             -0.006228
is_month_start   -0.004874
vol_mean_45      -0.003553
vol_mean_15      -0.003527
vol_90           -0.001801
sma_10           -0.000802
close            -0.000684
low              -0.000644
high             -0.000609
open             -0.000549
sma_3            -0.000463
sma_5            -0.000430
vol_15           -0.000391
mom_15           -0.000381
hl_range         -0.000082
sma_20            0.000046
vol_45            0.001418
mom_20            0.001937
ret_20            0.001937
vol_30            0.002054
mom_3             0.002181
vol3              0.002253
ret_3             0.003595
ret_1             0.003595
sma_ratio         0.004807
mom_5             0.004979
ret_5             0.004979
vol_5             0.005010
is_month_end      0.005310
dow               0.026872
target            1.000000
Name: target, dtype: float64

In [15]:
features = df.drop(columns=info+['target']).columns

In [16]:
df[df['ret_1']>1][['ticker', 'date', "close",'ret_1']]

,ticker,date,close,ret_1
80686,VTBR,2024-07-15,92.95,4662.823382


In [17]:
def smooth_outliers(df, features, lower_percentile=0.1, upper_percentile=0.9):
    """
    Функция для сглаживания выбросов в заданных фичах.
    Заменяет значения, которые выходят за пределы интервала между нижним и верхним персентилем.
    
    :param df: pandas DataFrame, исходный датасет.
    :param features: список названий колонок (фич), в которых нужно проверить выбросы.
    :param lower_percentile: нижний персентиль, по умолчанию 0.1 (10-й персентиль).
    :param upper_percentile: верхний персентиль, по умолчанию 0.9 (90-й персентиль).
    :return: pandas DataFrame с заменёнными выбросами.
    """
    
    data = df.copy()
    
    for feature in features:
        if feature in data.columns:
            # Вычисляем персентиль для нижней и верхней границы
            lower_value = data[feature].quantile(lower_percentile)
            upper_value = data[feature].quantile(upper_percentile)
            
            # Заменяем значения, выходящие за пределы интервала
            data[feature] = data[feature].apply(lambda x: min(max(x, lower_value), upper_value))
    
    return data

In [18]:
df = smooth_outliers(df, features, lower_percentile=0.01, upper_percentile=0.99)

In [19]:
df[df['ret_1']>1][['ticker', 'date', "close",'ret_1']]

,ticker,date,close,ret_1


In [20]:
path = f'prepared_data/dataset_for_feature_selection.pickle'

samples = {
    'df'           : df,
    'info_fields'  : info,
    'features'     : features,
    'cat_features' : [],
    'num_features' : features,
    'target'       : 'target',
}
with open(path, 'wb') as f:
    pickle.dump(samples, f)